In [17]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =====================================================================
# [Step 1] 금융 특화 5-Fold Time-Series Cross Validation (Purging & Embargo)
# =====================================================================
class PurgedEmbargoTimeSeriesCV:
    def __init__(self, n_splits=5, purge_window=5, embargo_window=21):
        self.n_splits = n_splits
        self.purge_window = purge_window     
        self.embargo_window = embargo_window 

    def split(self, X):
        n_samples = len(X)
        fold_bounds = np.linspace(0, n_samples, self.n_splits + 1, dtype=int)
        for i in range(self.n_splits):
            val_start = fold_bounds[i]
            val_end = fold_bounds[i + 1]
            val_indices = np.arange(val_start, val_end)
            
            train_pre_end = max(0, val_start - self.purge_window)
            train_pre_indices = np.arange(0, train_pre_end)
            
            train_post_start = min(n_samples, val_end + self.embargo_window)
            train_post_indices = np.arange(train_post_start, n_samples)
            
            train_indices = np.concatenate([train_pre_indices, train_post_indices])
            if len(train_indices) == 0:
                continue
            yield train_indices, val_indices


# =====================================================================
# [Step 2] 원하시는 8대 평가지표 정밀 추출 계량 엔진
# =====================================================================
def evaluate_textbook_remedies_perfect(X, y):
    X_with_const = sm.add_constant(X)
    n_samples, n_features = X_with_const.shape
    
    base_model = sm.OLS(y, X_with_const)
    base_res = base_model.fit()
    
    # 1. White Robust 표준오차 (이형산성 보정)
    res_white = base_model.fit(cov_type='HC3')
    
    # 2. WLS 모델 가동
    log_resid2 = np.log(base_res.resid**2)
    weight_var_model = sm.OLS(log_resid2, X_with_const).fit()
    weights = np.exp(weight_var_model.fittedvalues)
    wls_model = sm.WLS(y, X_with_const, weights=1.0 / weights)
    res_wls = wls_model.fit()
    
    # 3. GLSAR 모델 가동
    glsar_model = sm.GLSAR(y, X_with_const, rho=1)
    res_glsar = glsar_model.iterative_fit(maxiter=5)
    
    # 8대 지표 계산 부
    vifs = [variance_inflation_factor(X_with_const.values, i) for i in range(1, n_features)]
    mean_vif = np.mean(vifs) if len(vifs) > 0 else np.nan
    
    dw_stat = durbin_watson(base_res.resid)
    bp_p = het_breuschpagan(base_res.resid, X_with_const)[1]
    white_p = het_white(base_res.resid, X_with_const)[1] if X.shape[1] <= 15 else np.nan

    results_dict = {}
    results_dict['White_Robust_OLS'] = {
        'R2': res_white.rsquared, 'Adj_R2': res_white.rsquared_adj,
        'AIC': res_white.aic, 'BIC': res_white.bic,
        'Durbin_Watson': dw_stat, 'Breusch_Pagan_p': bp_p, 'White_p': white_p, 'Mean_VIF': mean_vif
    }
    results_dict['WLS_Model'] = {
        'R2': res_wls.rsquared, 'Adj_R2': res_wls.rsquared_adj,
        'AIC': res_wls.aic if hasattr(res_wls, 'aic') else base_res.aic, 
        'BIC': res_wls.bic if hasattr(res_wls, 'bic') else base_res.bic,
        'Durbin_Watson': durbin_watson(res_wls.resid), 
        'Breusch_Pagan_p': bp_p, 'White_p': white_p, 'Mean_VIF': mean_vif
    }
    results_dict['GLSAR_Model'] = {
        'R2': res_glsar.rsquared, 'Adj_R2': res_glsar.rsquared_adj,
        'AIC': res_glsar.aic if hasattr(res_glsar, 'aic') else base_res.aic,
        'BIC': res_glsar.bic if hasattr(res_glsar, 'bic') else base_res.bic,
        'Durbin_Watson': durbin_watson(res_glsar.resid), 
        'Breusch_Pagan_p': bp_p, 'White_p': white_p, 'Mean_VIF': mean_vif
    }
    return results_dict


if __name__ == "__main__":
    # 실제 데이터 연동 부
    spy = yf.download("SPY", start="2000-01-01", end="2026-06-30")
    available_cols = spy.columns
    if isinstance(available_cols, pd.MultiIndex):
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols.get_level_values(0) else spy['Close']
        spy_close = spy_close.iloc[:, 0]
    else:
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols else spy['Close']
        
    df = pd.DataFrame(index=spy_close.index)
    
    # [💡 7단원 해결 팁] 만약 자기상관(DW)을 완벽히 2.0으로 돌리고 싶다면 
    # 아래 주석을 해제하고 .pct_change(1)을 타겟으로 잡아보세요. 오버랩 오염이 완전히 치료됩니다.
    df['Target_Forward_Return'] = spy_close.pct_change(5).shift(-5)
    
    df['Lagged_Return_1D'] = spy_close.pct_change(1)
    delta = spy_close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    
    mid_band = spy_close.rolling(window=20).mean()
    std_dev = spy_close.rolling(window=20).std()
    upper_band = mid_band + (2 * std_dev)
    lower_band = mid_band - (2 * std_dev)
    df['Bollinger_BBP'] = (spy_close - lower_band) / (upper_band - lower_band)
    
    real_vol = spy_close.pct_change().rolling(20).std()
    vol_threshold_high = real_vol.quantile(0.66)
    vol_threshold_low = real_vol.quantile(0.33)
    regime = np.where(real_vol > vol_threshold_high, 'High_Vol', np.where(real_vol < vol_threshold_low, 'Low_Vol', 'Mid_Vol'))
    df_dummies = pd.get_dummies(regime, prefix='Regime', drop_first=True, dtype=float)
    df_dummies.index = df.index  
    df = pd.concat([df, df_dummies], axis=1)
    
    model1_features = ['Lagged_Return_1D', 'RSI_14', 'Bollinger_BBP'] + list(df_dummies.columns)
    ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')
    ff_df = ff_factors[0] / 100.0 
    df = df.join(ff_df, how='inner').dropna()
    
    fama_features = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
    model3_features = model1_features + fama_features
    train_data = df.loc["2000-01-01":"2020-12-31"]
    
    cv_splitter = PurgedEmbargoTimeSeriesCV(n_splits=5, purge_window=5, embargo_window=21)
    feature_sets = {"Model 1 (Technical)": model1_features, "Model 2 (Fama-French)": fama_features, "Model 3 (All Combined)": model3_features}
    master_report = {model_name: {'White_Robust_OLS': [], 'WLS_Model': [], 'GLSAR_Model': []} for model_name in feature_sets.keys()}
    
    print("===== [Step 1 & 2] 교재 Remedies 반영 및 Fold별 일수 정밀 추적 시작 =====")
    
    for fold, (train_idx, val_idx) in enumerate(cv_splitter.split(train_data), 1):
        # 💡 [해결 포인트] 로그에 각 Fold별 데이터 일수(len)를 정확하게 노출합니다.
        print(f"Fold {fold} 정밀 분석 중... (현재 학습 데이터 크기: {len(train_idx)}일 / 검증 데이터 크기: {len(val_idx)}일)")
        
        X_fold_train = train_data.iloc[train_idx]
        y_fold_train = train_data['Target_Forward_Return'].iloc[train_idx]
        
        for model_name, features in feature_sets.items():
            X_sub = X_fold_train[features]
            fold_results = evaluate_textbook_remedies_perfect(X_sub, y_fold_train)
            for remedy_name in ['White_Robust_OLS', 'WLS_Model', 'GLSAR_Model']:
                master_report[model_name][remedy_name].append(fold_results[remedy_name])

    print("\n==========================================================================================")
    print("   [최종 모델 및 교재 처방별 8대 평가지표 5-Fold CV 평균값 결과 표]")
    print("==========================================================================================")
    for model_name, remedies in master_report.items():
        print(f"\n★ 대상 독립변수군: {model_name}")
        remedy_summary = {}
        for remedy_name, metrics_list in remedies.items():
            remedy_summary[remedy_name] = pd.DataFrame(metrics_list).mean()
        df_summary = pd.DataFrame(remedy_summary).T
        col_order = ['R2', 'Adj_R2', 'AIC', 'BIC', 'Durbin_Watson', 'Breusch_Pagan_p', 'White_p', 'Mean_VIF']
        print(df_summary[col_order].to_string())

[*********************100%***********************]  1 of 1 completed
C:\Users\marji\AppData\Local\Temp\ipykernel_20524\1197450159.py:132: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')


===== [Step 1 & 2] 교재 Remedies 반영 및 Fold별 일수 정밀 추적 시작 =====
Fold 1 정밀 분석 중... (현재 학습 데이터 크기: 4191일 / 검증 데이터 크기: 1053일)
Fold 2 정밀 분석 중... (현재 학습 데이터 크기: 4186일 / 검증 데이터 크기: 1053일)
Fold 3 정밀 분석 중... (현재 학습 데이터 크기: 4186일 / 검증 데이터 크기: 1053일)
Fold 4 정밀 분석 중... (현재 학습 데이터 크기: 4186일 / 검증 데이터 크기: 1053일)
Fold 5 정밀 분석 중... (현재 학습 데이터 크기: 4207일 / 검증 데이터 크기: 1053일)

   [최종 모델 및 교재 처방별 8대 평가지표 5-Fold CV 평균값 결과 표]

★ 대상 독립변수군: Model 1 (Technical)
                     R2  Adj_R2     AIC     BIC  Durbin_Watson  Breusch_Pagan_p  White_p  Mean_VIF
White_Robust_OLS 0.0064  0.0052 -19,012 -18,974         0.4267           0.0000   0.0000    2.2702
WLS_Model        0.0058  0.0046 -20,454 -20,416         0.4387           0.0000   0.0000    2.2702
GLSAR_Model      0.3388  0.3380 -24,118 -24,080         0.2481           0.0000   0.0000    2.2702

★ 대상 독립변수군: Model 2 (Fama-French)
                     R2  Adj_R2     AIC     BIC  Durbin_Watson  Breusch_Pagan_p  White_p  Mean_VIF
White_Robust_OLS 0.0087  0.0075 -1

In [20]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =====================================================================
# [Step 1] 금융 특화 5-Fold Time-Series Cross Validation (Purging & Embargo)
# =====================================================================
class PurgedEmbargoTimeSeriesCV:
    def __init__(self, n_splits=5, purge_window=5, embargo_window=21):
        self.n_splits = n_splits
        self.purge_window = purge_window     
        self.embargo_window = embargo_window 

    def split(self, X):
        n_samples = len(X)
        fold_bounds = np.linspace(0, n_samples, self.n_splits + 1, dtype=int)
        for i in range(self.n_splits):
            val_start = fold_bounds[i]
            val_end = fold_bounds[i + 1]
            val_indices = np.arange(val_start, val_end)
            
            train_pre_end = max(0, val_start - self.purge_window)
            train_pre_indices = np.arange(0, train_pre_end)
            
            train_post_start = min(n_samples, val_end + self.embargo_window)
            train_post_indices = np.arange(train_post_start, n_samples)
            
            train_indices = np.concatenate([train_pre_indices, train_post_indices])
            if len(train_indices) == 0:
                continue
            yield train_indices, val_indices


# =====================================================================
# [Step 2] 순수 "일반 OLS" 평가지표 및 진단 엔진
# =====================================================================
def evaluate_pure_ols(X, y):
    """
    아무런 외래 Remedy 처방 없이, 가장 순수한 형태의 Ordinary Least Squares (OLS)
    추정치와 진단 통계량 8가지를 산출합니다.
    """
    X_with_const = sm.add_constant(X)
    n_samples, n_features = X_with_const.shape
    
    # 일반 OLS 모델 피팅
    model = sm.OLS(y, X_with_const)
    results = model.fit()
    
    metrics = {}
    # 1) R2 / 2) Adj_R2 / 3) AIC / 4) BIC
    metrics['R2'] = results.rsquared
    metrics['Adj_R2'] = results.rsquared_adj
    metrics['AIC'] = results.aic
    metrics['BIC'] = results.bic
    
    # 5) Durbin_Watson (자기상관)
    metrics['Durbin_Watson'] = durbin_watson(results.resid)
    
    # 6) Breusch_Pagan_p (이분산성 검정 1)
    bp_test = het_breuschpagan(results.resid, X_with_const)
    metrics['Breusch_Pagan_p'] = bp_test[1] 
    
    # 7) White_p (이분산성 검정 2 - 피처 수 제한 안전 장치)
    if X.shape[1] <= 15:
        white_test = het_white(results.resid, X_with_const)
        metrics['White_p'] = white_test[1]
    else:
        metrics['White_p'] = np.nan 
        
    # 8) Mean_VIF (다중공선성 평균값 - 상수항 제외)
    vifs = [variance_inflation_factor(X_with_const.values, i) for i in range(1, n_features)]
    metrics['Mean_VIF'] = np.mean(vifs) if len(vifs) > 0 else np.nan
    
    return metrics


# =====================================================================
# 메인 실제 데이터 다운로드 및 5-Fold CV 전용 파이프라인
# =====================================================================
if __name__ == "__main__":
    # 데이터 수집 (야후 파이낸스)
    spy = yf.download("SPY", start="2000-01-01", end="2026-06-30")
    
    available_cols = spy.columns
    if isinstance(available_cols, pd.MultiIndex):
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols.get_level_values(0) else spy['Close']
        spy_close = spy_close.iloc[:, 0]
    else:
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols else spy['Close']
        
    df = pd.DataFrame(index=spy_close.index)
    
    # [타겟 변수 설정] 1일 일별 수익률로 고정하여 직렬상관 원천 차단
    df['Target_Forward_Return'] = spy_close.pct_change(1).shift(-1)
    
    # 독립변수군 가공
    df['Lagged_Return_1D'] = spy_close.pct_change(1)
    delta = spy_close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    
    mid_band = spy_close.rolling(window=20).mean()
    std_dev = spy_close.rolling(window=20).std()
    upper_band = mid_band + (2 * std_dev)
    lower_band = mid_band - (2 * std_dev)
    df['Bollinger_BBP'] = (spy_close - lower_band) / (upper_band - lower_band)
    
    real_vol = spy_close.pct_change().rolling(20).std()
    vol_threshold_high = real_vol.quantile(0.66)
    vol_threshold_low = real_vol.quantile(0.33)
    regime = np.where(real_vol > vol_threshold_high, 'High_Vol', np.where(real_vol < vol_threshold_low, 'Low_Vol', 'Mid_Vol'))
    df_dummies = pd.get_dummies(regime, prefix='Regime', drop_first=True, dtype=float)
    df_dummies.index = df.index  
    df = pd.concat([df, df_dummies], axis=1)
    
    model1_features = ['Lagged_Return_1D', 'RSI_14', 'Bollinger_BBP'] + list(df_dummies.columns)
    ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')
    ff_df = ff_factors[0] / 100.0 
    df = df.join(ff_df, how='inner').dropna()
    
    fama_features = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
    model3_features = model1_features + fama_features
    train_data = df.loc["2000-01-01":"2020-12-31"]
    
    # 교차검증 구동
    cv_splitter = PurgedEmbargoTimeSeriesCV(n_splits=5, purge_window=5, embargo_window=21)
    feature_sets = {"Model 1 (Technical)": model1_features, "Model 2 (Fama-French)": fama_features, "Model 3 (All Combined)": model3_features}
    
    # 결과를 담을 레포트 초기화
    cv_summary_report = {model_name: [] for model_name in feature_sets.keys()}
    
    print("===== [Step 1 & 2] 순수 일반 OLS 기준 5-Fold 금융 교차검증 및 데이터 일수 로깅 시작 =====")
    
    for fold, (train_idx, val_idx) in enumerate(cv_splitter.split(train_data), 1):
        # ★ [일수 로그 수정 부] 각 폴드별 계산에 사용되는 정확한 거래일수를 화면에 인쇄합니다.
        print(f"Fold {fold} 정밀 분석 완료 -> [실제 학습 일수: {len(train_idx)}일 / 검증 일수: {len(val_idx)}일]")
        
        X_fold_train = train_data.iloc[train_idx]
        y_fold_train = train_data['Target_Forward_Return'].iloc[train_idx]
        
        for model_name, features in feature_sets.items():
            X_sub = X_fold_train[features]
            
            # 순수 일반 OLS 통계량 추출
            metrics = evaluate_pure_ols(X_sub, y_fold_train)
            cv_summary_report[model_name].append(metrics)

    # -----------------------------------------------------------------
    # 최종 평균값 표 리포트 출력
    # -----------------------------------------------------------------
    print("\n==========================================================================================")
    print("   [최종 순수 OLS 기준 8대 평가지표 5-Fold CV 평균값 결과 표]")
    print("==========================================================================================")
    
    final_comparison = {}
    for model_name in feature_sets.keys():
        df_metrics = pd.DataFrame(cv_summary_report[model_name])
        final_comparison[model_name] = df_metrics.mean()
        
    df_report = pd.DataFrame(final_comparison).T
    col_order = ['R2', 'Adj_R2', 'AIC', 'BIC', 'Durbin_Watson', 'Breusch_Pagan_p', 'White_p', 'Mean_VIF']
    print(df_report[col_order].to_string())

[*********************100%***********************]  1 of 1 completed
C:\Users\marji\AppData\Local\Temp\ipykernel_20524\754045061.py:124: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')


===== [Step 1 & 2] 순수 일반 OLS 기준 5-Fold 금융 교차검증 및 데이터 일수 로깅 시작 =====
Fold 1 정밀 분석 완료 -> [실제 학습 일수: 4191일 / 검증 일수: 1053일]
Fold 2 정밀 분석 완료 -> [실제 학습 일수: 4186일 / 검증 일수: 1053일]
Fold 3 정밀 분석 완료 -> [실제 학습 일수: 4186일 / 검증 일수: 1053일]
Fold 4 정밀 분석 완료 -> [실제 학습 일수: 4186일 / 검증 일수: 1053일]
Fold 5 정밀 분석 완료 -> [실제 학습 일수: 4207일 / 검증 일수: 1053일]

   [최종 순수 OLS 기준 8대 평가지표 5-Fold CV 평균값 결과 표]
                           R2  Adj_R2     AIC     BIC  Durbin_Watson  Breusch_Pagan_p  White_p  Mean_VIF
Model 1 (Technical)    0.0104  0.0092 -24,911 -24,873         2.0065           0.0000   0.0000    2.2702
Model 2 (Fama-French)  0.0131  0.0119 -24,923 -24,884         2.0009           0.0000   0.0000    1.3022
Model 3 (All Combined) 0.0140  0.0116 -24,916 -24,847         2.0046           0.0000   0.0000    9.4211


In [22]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web  # 실제 Fama-French 데이터 수집용
import yfinance as yf                  # 실제 S&P 500 가격 데이터 수집용
import statsmodels.api as sm

# =====================================================================
# [Step 1] 금융 특화 5-Fold Time-Series Cross Validation (Purging & Embargo)
# =====================================================================
class PurgedEmbargoTimeSeriesCV:
    """
    Marcos López de Prado 및 Stefan Jansen 교재 기반의 
    금융 시계열 교차검증 스플리터 (Purging & Embargo 완벽 구현) [cite: 134]
    """
    def __init__(self, n_splits=5, purge_window=5, embargo_window=21):
        self.n_splits = n_splits
        self.purge_window = purge_window     # 검증 전 구간 오염 방지 (Purge) [cite: 134]
        self.embargo_window = embargo_window # 검증 후 구간 자가상관 방지 (Embargo) [cite: 134]

    def split(self, X):
        n_samples = len(X)
        fold_bounds = np.linspace(0, n_samples, self.n_splits + 1, dtype=int)
        for i in range(self.n_splits):
            # 1. 검증 세트 (Validation Set) 구간 설정
            val_start = fold_bounds[i]
            val_end = fold_bounds[i + 1]
            val_indices = np.arange(val_start, val_end)
            
            # 2. 학습 세트 (Train Set) 인덱스 정의 (Purging 적용)
            train_pre_end = max(0, val_start - self.purge_window)
            train_pre_indices = np.arange(0, train_pre_end)
            
            # 3. 학습 세트 (Train Set) 인덱스 정의 (Embargo 적용)
            train_post_start = min(n_samples, val_end + self.embargo_window)
            train_post_indices = np.arange(train_post_start, n_samples)
            
            train_indices = np.concatenate([train_pre_indices, train_post_indices])
            if len(train_indices) == 0:
                continue
            yield train_indices, val_indices


# =====================================================================
# [Step 2] 7장 처방전 주입: 파마프렌치 요인별 진짜 p-value 추적 및 치유 엔진
# =====================================================================
def analyze_fama_remedy_p_values(X, y):
    """
    파마프렌치 5요인에 대해 교재의 처방을 OLS 모델 내부에 적용한 뒤,
    각 요인들의 p-value가 어떻게 수학적으로 완치되는지 추적합니다.
    """
    # statsmodels용 Intercept(상수항) 추가 [cite: 150]
    X_with_const = sm.add_constant(X)
    model = sm.OLS(y, X_with_const)
    
    # 1. 치료 전: 일반 OLS (이분산성/자기상관 노이즈에 노출되어 허풍을 떠는 통계)
    res_pure = model.fit()
    
    # 2. 처방 1 적용: White Robust 표준오차 (이분산성으로 인한 사기 p-value 완전 치료)
    res_white = model.fit(cov_type='HC3')
    
    # 3. 처방 2 적용: Newey-West 표준오차 (maxlags=1 적용하여 일별 직렬상관관계 완전 교정)
    res_newey = model.fit(cov_type='HAC', cov_kwds={'maxlags': 1})
    
    # 가독성 높은 비교 테이블 생성
    p_df = pd.DataFrame({
        '1_오염된_순수_OLS_p': res_pure.pvalues,
        '2_이분산성_치유(White)_p': res_white.pvalues,
        '3_자기상관_교정(Newey)_p': res_newey.pvalues
    })
    
    # 분석의 직관성을 위해 상수항(const)을 떼어내고 핵심 리스크 팩터만 노출
    p_df = p_df.drop('const', errors='ignore')
    return p_df


if __name__ == "__main__":
    print("=====================================================================")
    print("  야후 파이낸스 및 켄트 프렌치 라이브러리 연동 실제 데이터 다운로드")
    print("=====================================================================")
    
    spy = yf.download("SPY", start="2000-01-01", end="2026-06-30")
    available_cols = spy.columns
    if isinstance(available_cols, pd.MultiIndex):
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols.get_level_values(0) else spy['Close']
        spy_close = spy_close.iloc[:, 0]
    else:
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols else spy['Close']
        
    df = pd.DataFrame(index=spy_close.index)
    
    # 자기상관의 완벽한 처리를 위해 1일 일별 수익률을 예측 Y로 세팅
    df['Target_Forward_Return'] = spy_close.pct_change(1).shift(-1)
    
    # 실제 미국 켄 프렌치 데이터서버에서 파마프렌치 5요인 데이터 추출
    ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')
    ff_df = ff_factors[0] / 100.0 
    
    # 인덱스 실제 거래일 기준 교집합 결합
    df = df.join(ff_df, how='inner').dropna()
    
    # Model 2 전용 실제 Fama-French 5요인 독립변수 목록 [cite: 143]
    fama_features = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
    
    # 인샘플 훈련 구간 슬라이싱 (2000년 ~ 2020년)
    train_data = df.loc["2000-01-01":"2020-12-31"]
    
    # 5-Fold 교차검증 스플리터 선언 [cite: 134]
    cv_splitter = PurgedEmbargoTimeSeriesCV(n_splits=5, purge_window=5, embargo_window=21)
    
    print("\n===== [Model 2 파마프렌치 전용] 7장 해결법 주입 및 Fold별 일수 로깅 시작 =====")
    
    fold_p_reports = []
    
    for fold, (train_idx, val_idx) in enumerate(cv_splitter.split(train_data), 1):
        # 💡 [요청사항 완벽 해결] 각 폴드별 퍼징/엠바고 처리가 완료된 실제 데이터 일수를 인쇄
        print(f"\n▶ Fold {fold} 검증 가동 -> [실제 학습 일수: {len(train_idx)}일 / 검증 일수: {len(val_idx)}일]")
        
        X_fold_train = train_data.iloc[train_idx][fama_features]
        y_fold_train = train_data['Target_Forward_Return'].iloc[train_idx]
        
        # 파마프렌치 요인별 7장 치유식 가동
        p_report = analyze_fama_remedy_p_values(X_fold_train, y_fold_train)
        fold_p_reports.append(p_report)
        
        # 실시간 변수 완치 현황 화면 인쇄
        print(p_report.round(4).to_string())

    print("\n==========================================================================================")
    print("   [최종 Model 2 전체 5-Fold CV 평균: 오염 상태 vs 7장 완치 처방 후 진짜 변수 p-value 비교 표]")
    print("==========================================================================================")
    df_final_p = pd.concat(fold_p_reports).groupby(level=0).mean()
    print(df_final_p.round(4).to_string())

[*********************100%***********************]  1 of 1 completed

  야후 파이낸스 및 켄트 프렌치 라이브러리 연동 실제 데이터 다운로드



C:\Users\marji\AppData\Local\Temp\ipykernel_20524\2118730112.py:95: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')



===== [Model 2 파마프렌치 전용] 7장 해결법 주입 및 Fold별 일수 로깅 시작 =====

▶ Fold 1 검증 가동 -> [실제 학습 일수: 4207일 / 검증 일수: 1056일]
        1_오염된_순수_OLS_p  2_이분산성_치유(White)_p  3_자기상관_교정(Newey)_p
Mkt-RF          0.0000              0.0023              0.0023
SMB             0.0000              0.0253              0.0198
HML             0.5297              0.7667              0.7586
RMW             0.9718              0.9828              0.9828
CMA             0.7064              0.8329              0.8318

▶ Fold 2 검증 가동 -> [실제 학습 일수: 4201일 / 검증 일수: 1057일]
        1_오염된_순수_OLS_p  2_이분산성_치유(White)_p  3_자기상관_교정(Newey)_p
Mkt-RF          0.0000              0.0045              0.0051
SMB             0.0004              0.0615              0.0549
HML             0.3211              0.6081              0.5977
RMW             0.8398              0.8947              0.8951
CMA             0.1443              0.3478              0.3497

▶ Fold 3 검증 가동 -> [실제 학습 일수: 4201일 / 검증 일수: 1057일]
        1_오염된_순수_OLS_p  2_이분산

In [23]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import statsmodels.api as sm
import pickle
from sklearn.preprocessing import StandardScaler

# =====================================================================
# [Step 1] 금융 특화 5-Fold Time-Series Cross Validation (Purging & Embargo)
# =====================================================================
class PurgedEmbargoTimeSeriesCV:
    def __init__(self, n_splits=5, purge_window=5, embargo_window=21):
        self.n_splits = n_splits
        self.purge_window = purge_window     
        self.embargo_window = embargo_window 

    def split(self, X):
        n_samples = len(X)
        fold_bounds = np.linspace(0, n_samples, self.n_splits + 1, dtype=int)
        for i in range(self.n_splits):
            val_start = fold_bounds[i]
            val_end = fold_bounds[i + 1]
            val_indices = np.arange(val_start, val_end)
            
            train_pre_end = max(0, val_start - self.purge_window)
            train_pre_indices = np.arange(0, train_pre_end)
            
            train_post_start = min(n_samples, val_end + self.embargo_window)
            train_post_indices = np.arange(train_post_start, n_samples)
            
            train_indices = np.concatenate([train_pre_indices, train_post_indices])
            if len(train_indices) == 0:
                continue
            yield train_indices, val_indices


# =====================================================================
# [Step 2] 7장 처방전 주입: 파마프렌치 요인별 p-value 정밀 교정 엔진
# =====================================================================
def analyze_fama_remedy_p_values(X, y):
    """
    파마프렌치 5요인에 대해 교재 처방을 OLS 모델 내부에 완벽히 주입하여
    이분산성(White)과 자기상관(Newey-West) 오염을 교정한 p-value 테이블을 리턴합니다.
    """
    X_with_const = sm.add_constant(X)
    model = sm.OLS(y, X_with_const)
    
    res_pure = model.fit()
    res_white = model.fit(cov_type='HC3')
    res_newey = model.fit(cov_type='HAC', cov_kwds={'maxlags': 1})
    
    p_df = pd.DataFrame({
        '1_오염된_순수_OLS_p': res_pure.pvalues,
        '2_이분산성_치유(White)_p': res_white.pvalues,
        '3_자기상관_교정(Newey)_p': res_newey.pvalues
    })
    
    p_df = p_df.drop('const', errors='ignore')
    return p_df


# =====================================================================
# 🚀 메인 제어 파이프라인 (4단계 전체 통합 및 영구 저장 실행)
# =====================================================================
if __name__ == "__main__":
    print("=====================================================================")
    print("  [4단계 통합 파이프라인] Model 2 파마프렌치 5요인 실전 데이터 연동 및 기동")
    print("=====================================================================")
    
    # [1단계 & 2단계] 실제 시장 데이터 수집 및 교차검증용 가공
    print("-> S&P 500(SPY) 일별 가격 데이터 수집 중...")
    spy = yf.download("SPY", start="2000-01-01", end="2026-06-30")
    
    available_cols = spy.columns
    if isinstance(available_cols, pd.MultiIndex):
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols.get_level_values(0) else spy['Close']
        spy_close = spy_close.iloc[:, 0]
    else:
        spy_close = spy['Adj Close'] if 'Adj Close' in available_cols else spy['Close']
        
    df = pd.DataFrame(index=spy_close.index)
    
    # 💡 [직렬상관 원천 차단] 1일 일별 수익률을 예측 Y로 고정 (Durbin-Watson 2.0 달성)
    df['Target_Forward_Return'] = spy_close.pct_change(1).shift(-1)
    
    print("-> Fama-French 5 Factors 실제 리스크 요인 데이터 수집 중...")
    ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')
    ff_df = ff_factors[0] / 100.0 
    
    # 두 실제 데이터의 거래일 교집합 Inner Join 병합 및 결측치 드랍
    df_full = df.join(ff_df, how='inner').dropna()
    
    fama_features = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
    train_data = df_full.loc["2000-01-01":"2020-12-31"]
    
    # [3단계] Purging & Embargo 교차검증 구동 및 폴드별 실제 일수 로깅
    print("\n===== [검증 단계] 7장 해결법 주입 및 Fold별 학습 일수 로깅 시작 =====")
    cv_splitter = PurgedEmbargoTimeSeriesCV(n_splits=5, purge_window=5, embargo_window=21)
    fold_p_reports = []
    
    for fold, (train_idx, val_idx) in enumerate(cv_splitter.split(train_data), 1):
        # 💡 각 폴드 실행 시 정보 누수가 차단된 순수 연산 일수(len)를 완벽히 출력합니다.
        print(f"\n▶ Fold {fold} 분석 가동 -> [실제 학습 일수: {len(train_idx)}일 / 검증 일수: {len(val_idx)}일]")
        
        X_fold_train = train_data.iloc[train_idx][fama_features]
        y_fold_train = train_data['Target_Forward_Return'].iloc[train_idx]
        
        p_report = analyze_fama_remedy_p_values(X_fold_train, y_fold_train)
        fold_p_reports.append(p_report)
        print(p_report.round(4).to_string())

    print("\n==========================================================================================")
    print("   [교차검증 최종 요약: 오염 상태 vs 7장 완치 처방 후 진짜 변수 p-value 비교 표]")
    print("==========================================================================================")
    df_final_p = pd.concat(fold_p_reports).groupby(level=0).mean()
    print(df_final_p.round(4).to_string())
    
    # [4단계] 2000년 ~ 2026년 전체 데이터 기반 통합 최종 학습 및 파일 디스크 영구 저장
    print("\n==========================================================================================")
    print("   [최종 단계] 2000~2026 전체 데이터 기준 통합 모델 피팅 및 피클(.pkl) 파일 저장 시작")
    print("==========================================================================================")
    
    X_full = df_full[fama_features]
    y_full = df_full['Target_Forward_Return']
    
    # 실전 서빙을 위해 독립변수 전체 정규화(Standardization) 및 스케일러 보관
    scaler = StandardScaler()
    X_full_scaled = pd.DataFrame(scaler.fit_transform(X_full), columns=X_full.columns, index=X_full.index)
    
    # OLS 내부 상수항 주입 및 피팅
    X_full_scaled_const = sm.add_constant(X_full_scaled)
    final_model_core = sm.OLS(y_full, X_full_scaled_const)
    
    # 7장 최종 처방: 이분산성 분산 왜곡을 원천 차단한 강력한 최종 Robust 계수 산출 (HC3)
    final_model_fit = final_model_core.fit(cov_type='HC3')
    
    print("\n[완료] 이분산성이 완전히 치료된 Model 2의 26년 전체 데이터 최종 파라미터 요약")
    print(final_model_fit.summary())
    
    # 고성능 바이너리 파일 영구 저장
    model_filename = "fama_french_best_model.pkl"
    scaler_filename = "fama_french_scaler.pkl"
    
    with open(model_filename, 'wb') as m_file:
        pickle.dump(final_model_fit, m_file)
        
    with open(scaler_filename, 'wb') as s_file:
        pickle.dump(scaler, s_file)
        
    print("\n=====================================================================")
    print(f"  💾 파이프라인 영구 보관 완료!")
    print(f"  - 모델 바이너리 파일명: '{model_filename}'")
    print(f"  - 스케일러 바이너리 파일명: '{scaler_filename}'")
    print("=====================================================================")

[*********************100%***********************]  1 of 1 completed

  [4단계 통합 파이프라인] Model 2 파마프렌치 5요인 실전 데이터 연동 및 기동
-> S&P 500(SPY) 일별 가격 데이터 수집 중...
-> Fama-French 5 Factors 실제 리스크 요인 데이터 수집 중...



C:\Users\marji\AppData\Local\Temp\ipykernel_20524\219631357.py:88: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2000-01-01', end='2026-06-30')



===== [검증 단계] 7장 해결법 주입 및 Fold별 학습 일수 로깅 시작 =====

▶ Fold 1 분석 가동 -> [실제 학습 일수: 4207일 / 검증 일수: 1056일]
        1_오염된_순수_OLS_p  2_이분산성_치유(White)_p  3_자기상관_교정(Newey)_p
Mkt-RF          0.0000              0.0023              0.0023
SMB             0.0000              0.0253              0.0198
HML             0.5297              0.7667              0.7586
RMW             0.9718              0.9828              0.9828
CMA             0.7064              0.8329              0.8318

▶ Fold 2 분석 가동 -> [실제 학습 일수: 4201일 / 검증 일수: 1057일]
        1_오염된_순수_OLS_p  2_이분산성_치유(White)_p  3_자기상관_교정(Newey)_p
Mkt-RF          0.0000              0.0045              0.0051
SMB             0.0004              0.0615              0.0549
HML             0.3211              0.6081              0.5977
RMW             0.8398              0.8947              0.8951
CMA             0.1443              0.3478              0.3497

▶ Fold 3 분석 가동 -> [실제 학습 일수: 4201일 / 검증 일수: 1057일]
        1_오염된_순수_OLS_p  2_이분산성_치유(Whi